# Agentic Knowledge Graph over PubMed (40M architecture, laptop demo)

A **knowledge-graph RAG** built from structured metadata rather than LLM extraction.
This notebook is a self-contained, CPU-only recreation of the architecture used to
index the PubMed 2026 baseline (~40 million records) as a citation + MeSH graph.

It does **not** download the 50 GiB baseline or call a 14B model. Instead it runs
the same stages on a tiny in-memory corpus so every cell finishes in seconds:

1. Parse records into article / citation / MeSH tables (zero extraction calls)
2. Size the graph (induced-subgraph retention)
3. Build a MeSH ontology with string-prefix ancestry
4. Store edges as compressed sparse row (CSR) adjacency
5. Ground a question with longest-match lookup (no model emits an identifier)
6. Walk DIRECT co-annotation and one-hop citation BRIDGE paths
7. Refuse before generation when the graph has no usable path
8. Rank terminals, form a yes / no / maybe posterior, and abstain if unconfident

Published full-corpus numbers below are from the public case study
([Fareed Khan, 2026](https://github.com/FareedKhan-dev/agentic-knowledge-graph))
and are shown as **reference**, not as output of this run.

| published property | value |
|---|---|
| PubMed 2026 baseline files | 1,334 gzipped XML |
| articles parsed | 39,994,988 |
| edges parsed | 929,824,202 |
| LLM calls to build the graph | 0 |
| held-out PubMedQA (calibrated) | 83.2% (n = 600) |

Pipeline modules this notebook previews: `parse_pubmed.py`, `build_graph4.py`,
`embed_corpus3.py`, `consolidate_index.py`, `kg_ground3.py` … `kg_ground6.py`.



## 0. Parameters and environment

Every random source is seeded. GPU is optional; the demo is written to run on CPU.



In [ ]:
from __future__ import annotations

import itertools
import math
import random
import re
import time
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Iterable

import numpy as np

SEED = 0
rng = np.random.default_rng(SEED)
random.seed(SEED)

MIN_PATHS = 1
MAX_PATHS = 12
MAX_SEED = 40
CHECK_TAG_MAX = 8
COMMON_SINGLE = {
    "risk", "role", "results", "effect", "effects", "decrease", "increase",
    "influence", "affect", "useful", "effective", "diagnosis", "treatment",
    "study", "patients", "disease", "against",
}

print(f"numpy {np.__version__}")
print(f"seed  {SEED}")

try:
    import torch
    print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        x = torch.randn(1024, 1024, device="cuda", dtype=torch.float16)
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(8):
            _ = x @ x
        torch.cuda.synchronize()
        tflops = 8 * 2 * 1024**3 / (time.time() - t0) / 1e12
        print(f"fp16 matmul  {tflops:.2f} TFLOP/s  device={torch.cuda.get_device_name(0)}")
except Exception as exc:
    print(f"torch not required for this demo ({type(exc).__name__})")


## 1. A corpus that already ships its edges

PubMed is the architecture: tens of millions of free records, curator-resolved
citation PMIDs, and human/classifier-assigned MeSH headings. The graph is a
**parse**, not an extraction.

The demo corpus is small on purpose. Each record still has the same five
families the 40M parser writes: articles, MeSH edges, citations, corrections,
and publication types.



In [ ]:
# Mini MEDLINE-shaped records. Every field is authored metadata, not model output.
# PMIDs are fictional and reserved for this notebook.

MESH = {
    "D001241": {"name": "Aspirin", "tree": ["D03.633.100.473.402.069"], "cat": "D", "depth": 5,
                "terms": ["aspirin", "acetylsalicylic acid"]},
    "D009203": {"name": "Myocardial Infarction", "tree": ["C14.280.647.500", "C14.907.585.500"],
                "cat": "C", "depth": 4,
                "terms": ["myocardial infarction", "heart attack", "infarction, myocardial"]},
    "D008687": {"name": "Metformin", "tree": ["D02.078.370.120.425"], "cat": "D", "depth": 5,
                "terms": ["metformin"]},
    "D003924": {"name": "Diabetes Mellitus, Type 2", "tree": ["C18.452.394.750.149"],
                "cat": "C", "depth": 5,
                "terms": ["type 2 diabetes", "diabetes mellitus type 2", "niddm"]},
    "D006886": {"name": "Hydroxychloroquine", "tree": ["D03.633.300.240.380"], "cat": "D", "depth": 5,
                "terms": ["hydroxychloroquine"]},
    "D000086382": {"name": "COVID-19", "tree": ["C01.925.782.600.550.200.170"], "cat": "C", "depth": 6,
                   "terms": ["covid-19", "covid 19", "sars-cov-2 infection"]},
    "D001471": {"name": "Barrett Esophagus", "tree": ["C06.405.117.102"], "cat": "C", "depth": 4,
                "terms": ["barrett esophagus", "barrett oesophagus", "barrett's esophagus"]},
    "D007633": {"name": "Keratins", "tree": ["D05.750.078.593.450", "D12.776.860.450"],
                "cat": "D", "depth": 4, "terms": ["keratins", "cytokeratin", "cytokeratins"]},
    "D000758": {"name": "Anesthesia", "tree": ["E03.155"], "cat": "E", "depth": 2,
                "terms": ["anesthesia", "anaesthesia", "pediatric anesthesia",
                          "paediatric anaesthesia"]},
    "D010372": {"name": "Pediatrics", "tree": ["H02.403.670"], "cat": "H", "depth": 3,
                "terms": ["pediatrics", "paediatrics", "pediatric", "paediatric"]},
    "D002446": {"name": "Celiac Disease", "tree": ["C06.405.205.249"], "cat": "C", "depth": 4,
                "terms": ["celiac disease", "coeliac disease"]},
    "D002944": {"name": "Circumcision, Male", "tree": ["E04.950.774.150"], "cat": "E", "depth": 4,
                "terms": ["circumcision, male", "male circumcision"]},
    "D014409": {"name": "Tumor Necrosis Factor-alpha", "tree": ["D12.644.276.374.500"],
                "cat": "D", "depth": 5,
                "terms": ["tumor necrosis factor", "tumour necrosis factor", "tnf-alpha"]},
    "D000818": {"name": "Animals", "tree": ["B01.050"], "cat": "B", "depth": 2,
                "terms": ["animals"]},
    "D006801": {"name": "Humans", "tree": ["M01.390"], "cat": "M", "depth": 2,
                "terms": ["humans"]},
    "D000328": {"name": "Adult", "tree": ["M01.060.116"], "cat": "M", "depth": 3,
                "terms": ["adult"]},
}

# Supplementary concept: pembrolizumab is an alias onto a main descriptor, not a new node.
SCR = {
    "C000613208": {
        "name": "pembrolizumab",
        "mapped_to": ["D014409"],  # demo alias onto TNF superfamily neighbourhood
        "terms": ["pembrolizumab", "keytruda"],
    }
}

ARTICLES = [
    dict(pmid=1001, year=2020, retracted=False, pubtype="Journal Article",
         title="Aspirin reduces first myocardial infarction in high-risk adults",
         abstract=("A randomized trial of daily aspirin versus placebo found a clear reduction "
                   "in first myocardial infarction. The authors conclude that aspirin is effective "
                   "for primary prevention in the studied high-risk adults."),
         mesh=[("D001241", True), ("D009203", True), ("D006801", False), ("D000328", False)]),
    dict(pmid=1002, year=2018, retracted=False, pubtype="Review",
         title="Aspirin for primary prevention: a narrative review",
         abstract=("This review summarises trials of aspirin for preventing myocardial infarction. "
                   "Net benefit depends on bleeding risk. Overall the evidence supports a modest "
                   "reduction in infarction events."),
         mesh=[("D001241", True), ("D009203", True), ("D006801", False)]),
    dict(pmid=1003, year=2015, retracted=False, pubtype="Journal Article",
         title="Pathophysiology of myocardial infarction",
         abstract=("Myocardial infarction follows acute coronary occlusion. This paper describes "
                   "ischaemic injury and does not evaluate aspirin."),
         mesh=[("D009203", True), ("D006801", False)]),
    dict(pmid=1004, year=2019, retracted=False, pubtype="Journal Article",
         title="Metformin as first-line therapy in type 2 diabetes mellitus",
         abstract=("Metformin lowered HbA1c versus placebo in adults with type 2 diabetes. "
                   "The trial concludes metformin remains effective first-line treatment."),
         mesh=[("D008687", True), ("D003924", True), ("D006801", False)]),
    dict(pmid=1005, year=2017, retracted=False, pubtype="Guideline",
         title="Type 2 diabetes mellitus treatment guidelines",
         abstract=("Guidelines recommend metformin as initial pharmacologic therapy for most "
                   "adults with type 2 diabetes mellitus unless contraindicated."),
         mesh=[("D003924", True), ("D008687", False), ("D006801", False)]),
    dict(pmid=1006, year=2020, retracted=False, pubtype="Journal Article",
         title="Hydroxychloroquine in hospitalized COVID-19: a negative trial",
         abstract=("Hydroxychloroquine did not improve clinical status in hospitalized COVID-19. "
                   "The authors conclude hydroxychloroquine is not effective against COVID-19 "
                   "in this population."),
         mesh=[("D006886", True), ("D000086382", True), ("D006801", False)]),
    dict(pmid=1007, year=2020, retracted=False, pubtype="Review",
         title="Clinical features of COVID-19",
         abstract=("COVID-19 is the disease caused by SARS-CoV-2. This review covers presentation "
                   "and does not evaluate hydroxychloroquine."),
         mesh=[("D000086382", True), ("D006801", False)]),
    dict(pmid=1008, year=2020, retracted=True, pubtype="Journal Article",
         title="RETRACTED: hydroxychloroquine cures COVID-19",
         abstract=("This record was retracted. It previously claimed hydroxychloroquine was "
                   "effective against COVID-19. Do not use as evidence."),
         mesh=[("D006886", True), ("D000086382", True)]),
    dict(pmid=1009, year=2016, retracted=False, pubtype="Journal Article",
         title="Cytokeratin immunoreactivity in Barrett esophagus",
         abstract=("Cytokeratin staining distinguished Barrett esophagus from gastric mucosa. "
                   "The authors conclude cytokeratin immunoreactivity is useful in the diagnosis "
                   "of Barrett esophagus."),
         mesh=[("D001471", True), ("D007633", True)]),
    dict(pmid=1010, year=2021, retracted=False, pubtype="Review",
         title="Paediatric anaesthesia safety update",
         abstract=("A review of paediatric anaesthesia practice. No comparative effectiveness "
                   "claim is made."),
         mesh=[("D000758", True), ("D010372", True), ("D006801", False)]),
    dict(pmid=1011, year=2014, retracted=False, pubtype="Journal Article",
         title="Serology for coeliac disease diagnosis",
         abstract=("Tissue transglutaminase antibodies support the diagnosis of coeliac disease "
                   "in symptomatic children and adults."),
         mesh=[("D002446", True), ("D006801", False)]),
    dict(pmid=1012, year=2013, retracted=False, pubtype="Journal Article",
         title="Outcomes after male circumcision",
         abstract=("Male circumcision reduced some infection risks in the studied cohorts. "
                   "The finding is observational."),
         mesh=[("D002944", True), ("D006801", False)]),
    dict(pmid=1013, year=2018, retracted=False, pubtype="Journal Article",
         title="Tumour necrosis factor blockade in rheumatoid arthritis",
         abstract=("TNF-alpha inhibition improved joint scores. Pembrolizumab is mentioned only "
                   "as a related immunotherapy, not as an RA treatment."),
         mesh=[("D014409", True), ("D006801", False)]),
    dict(pmid=1014, year=1990, retracted=False, pubtype="Journal Article",
         title="An early note on salicylates and coronary events",
         abstract="",  # not quotable
         mesh=[("D001241", True), ("D009203", False)]),
    dict(pmid=1015, year=2022, retracted=False, pubtype="Journal Article",
         title="Pembrolizumab in advanced melanoma",
         abstract=("Pembrolizumab prolonged survival versus chemotherapy in advanced melanoma. "
                   "The trial concludes pembrolizumab is effective in this setting."),
         mesh=[("D014409", False), ("D006801", False)]),
]

CITATIONS = [
    (1001, 1002), (1001, 1003), (1001, 1014),
    (1002, 1003), (1002, 1014),
    (1004, 1005),
    (1006, 1007), (1006, 1008),
    (1009, 1011),
    (1013, 1015),
    (1015, 1013),
]

CORRECTIONS = [
    (1008, 1008, "RetractionIn"),
]


def parse_records(articles, citations, corrections):
    tables = {k: [] for k in ("articles", "mesh_edges", "citations", "corrections", "pubtypes")}
    for a in articles:
        tables["articles"].append({
            "pmid": a["pmid"], "year": a["year"], "title": a["title"],
            "abstract": a["abstract"], "retracted": a["retracted"],
            "quotable": bool(a["abstract"].strip()),
        })
        tables["pubtypes"].append((a["pmid"], a["pubtype"]))
        for ui, major in a["mesh"]:
            tables["mesh_edges"].append((a["pmid"], ui, major, None))
    tables["citations"].extend(citations)
    tables["corrections"].extend(corrections)
    return tables


T = parse_records(ARTICLES, CITATIONS, CORRECTIONS)
n_nodes = len(T["articles"])
n_edges = (len(T["mesh_edges"]) + len(T["citations"]) + len(T["corrections"]) + len(T["pubtypes"]))
print("table            rows")
print("----------------------")
for name, rows in T.items():
    print(f"{name:<16} {len(rows):>4}")
print("----------------------")
print(f"NODES            {n_nodes}")
print(f"EDGES            {n_edges}   (demo parse; published 40M parse = 929,824,202)")
print("LLM calls used to build any of this: 0")


## 2. Why the corpus has to be the full baseline

A citation edge is usable only if **both** endpoints sit inside the indexed set.
A recent slice looks large and still drops most induced edges. The published
study measured 28.1% retention on a recent 10M slice versus ~100% on the full
40M baseline. The demo repeats that measurement on the toy graph and on the
published slice table.



In [ ]:
pmids = {a["pmid"] for a in T["articles"]}
citing_inside = [(u, v) for u, v in T["citations"] if u in pmids]
induced = [(u, v) for u, v in citing_inside if v in pmids]
print(f"total citation edges       {len(T['citations'])}")
print(f"cited_pmid inside corpus   {len(induced)}   ({100 * len(induced) / max(len(T['citations']), 1):.1f}%)")
print(f"dangling                   {len(citing_inside) - len(induced)}")

# Published retention numbers (reference, not computed here).
published_slices = [
    ("recent  5M", 10.5, 58.0),
    ("recent 10M", 28.1, 63.6),
    ("recent 20M", 61.7, 72.4),
    ("FULL   40M", 100.0, None),
]
print()
print("published slice retention (Fareed Khan 2026, full baseline)")
print(f"{'slice':<12} {'retained':>10} {'MeSH cov':>10}")
print("-" * 34)
for name, ret, mesh in published_slices:
    mesh_s = f"{mesh:.1f}%" if mesh is not None else "   —"
    print(f"{name:<12} {ret:>9.1f}% {mesh_s:>10}")

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is not None:
    fig, ax = plt.subplots(figsize=(6.2, 3.4))
    labels = [r[0] for r in published_slices]
    vals = [r[1] for r in published_slices]
    colors = ["#8a8880", "#8a8880", "#8a8880", "#2a78d6"]
    ax.bar(labels, vals, color=colors)
    ax.set_ylabel("induced citation retention (%)")
    ax.set_title("Why a recent slice is the wrong graph")
    ax.set_ylim(0, 110)
    for i, v in enumerate(vals):
        ax.text(i, v + 2, f"{v:.1f}%", ha="center", fontsize=9)
    fig.tight_layout()
    plt.show()


## 3. MeSH is the ontology we did not have to build

Tree numbers are dotted paths, so ancestry is a **string-prefix** test. No
reasoner, no Datalog. Supplementary concept records are curated aliases onto
descriptors that already exist — they must not become a new node type.



In [ ]:
name_of = {ui: rec["name"] for ui, rec in MESH.items()}
ui2tn = {ui: rec["tree"] for ui, rec in MESH.items()}
tn2ui = {tn: ui for ui, rec in MESH.items() for tn in rec["tree"]}
# Ancestor labels for prefixes that are not themselves demo descriptors.
TREE_NAME = {
    "C14": "Cardiovascular Diseases",
    "C14.280": "Heart Diseases",
    "C14.280.647": "Myocardial Ischemia",
    "C14.280.647.500": "Myocardial Infarction",
}

print(f"mesh_descriptors   {len(MESH)}")
print(f"mesh_tree          {sum(len(r['tree']) for r in MESH.values())}")
print(f"mesh_terms         {sum(len(r['terms']) for r in MESH.values())}")
print(f"supplementary      {len(SCR)}")

probe = next(ui for ui, rec in MESH.items() if rec["name"] == "Myocardial Infarction")
print(f"\n'{name_of[probe]}' ({probe})  tree numbers: {ui2tn[probe]}")
print("ancestry by STRING PREFIX — no graph walk:")
parts = ui2tn[probe][0].split(".")
for i in range(1, len(parts) + 1):
    anc = ".".join(parts[:i])
    label = TREE_NAME.get(anc) or name_of.get(tn2ui.get(anc, ""), "?")
    print(f"    {anc:<22} {label}")


## 4. The graph store is CSR, not a graph database

Neighbours of node `i` are `indices[indptr[i]:indptr[i+1]]`. Both directions
are built. MeSH→article arrays and the major-topic flags are derived from
**one** `lexsort` so they cannot silently permute against each other
(`build_graph4.py` exists because an earlier two-pass build did exactly that).



In [ ]:
def build_csr(src: np.ndarray, dst: np.ndarray, n_nodes: int):
    counts = np.bincount(src, minlength=n_nodes).astype(np.int64)
    indptr = np.zeros(n_nodes + 1, dtype=np.int64)
    np.cumsum(counts, out=indptr[1:])
    order = np.argsort(src, kind="stable")
    return indptr, dst[order].astype(np.int32)


class KGStore:
    def __init__(self, articles, citations, mesh_edges, mesh_meta):
        t0 = time.time()
        self.pmids = np.array(sorted(a["pmid"] for a in articles), dtype=np.int64)
        self.N = len(self.pmids)
        self._pmid_to_i = {int(p): i for i, p in enumerate(self.pmids)}
        self.year = np.zeros(self.N, dtype=np.int16)
        self.retracted = np.zeros(self.N, dtype=np.bool_)
        self.quotable = np.zeros(self.N, dtype=np.bool_)
        self.title = [""] * self.N
        self.abstract = [""] * self.N
        by_pmid = {a["pmid"]: a for a in articles}
        for i, p in enumerate(self.pmids):
            a = by_pmid[int(p)]
            self.year[i] = a["year"]
            self.retracted[i] = a["retracted"]
            self.quotable[i] = a["quotable"]
            self.title[i] = a["title"]
            self.abstract[i] = a["abstract"]

        src, dst = [], []
        for u, v in citations:
            if u in self._pmid_to_i and v in self._pmid_to_i:
                src.append(self._pmid_to_i[u])
                dst.append(self._pmid_to_i[v])
        src, dst = np.asarray(src, np.int64), np.asarray(dst, np.int64)
        self.cite_indptr, self.cite_indices = build_csr(src, dst, self.N)
        self.cb_indptr, self.cb_indices = build_csr(dst, src, self.N)

        self.mesh_uis = np.array(sorted(mesh_meta), dtype=object)
        self.D = len(self.mesh_uis)
        self._ui_to_d = {str(u): d for d, u in enumerate(self.mesh_uis)}
        self.mesh_name = [mesh_meta[str(u)]["name"] for u in self.mesh_uis]
        self.mesh_cat = np.array([mesh_meta[str(u)]["cat"] for u in self.mesh_uis])
        self.mesh_min_depth = np.array([mesh_meta[str(u)]["depth"] for u in self.mesh_uis], np.int16)

        pairs = {}
        for pmid, ui, major, _qual in mesh_edges:
            if pmid not in self._pmid_to_i or ui not in self._ui_to_d:
                continue
            key = (self._pmid_to_i[pmid], self._ui_to_d[ui])
            pairs[key] = pairs.get(key, False) or bool(major)
        src_m = np.array([a for (a, _d) in pairs], dtype=np.int64)
        dst_m = np.array([d for (_a, d) in pairs], dtype=np.int64)
        maj = np.array([pairs[(int(a), int(d))] for a, d in zip(src_m, dst_m)], dtype=np.bool_)

        # One sorted frame → indices and major flags cannot disagree.
        order = np.lexsort((src_m, dst_m))
        d_src, d_dst, d_maj = src_m[order], dst_m[order], maj[order]
        cnt = np.bincount(d_dst, minlength=self.D).astype(np.int64)
        self.m2a_indptr = np.zeros(self.D + 1, dtype=np.int64)
        np.cumsum(cnt, out=self.m2a_indptr[1:])
        self.m2a_indices = d_src.astype(np.int32)
        self.m2a_major = d_maj
        assert len(self.m2a_indices) == len(self.m2a_major) == int(self.m2a_indptr[-1])

        self.mesh_count = np.diff(self.m2a_indptr).astype(np.int32)
        self.load_secs = time.time() - t0

    def idx(self, pmid: int) -> int:
        return self._pmid_to_i.get(int(pmid), -1)

    def pmid(self, i: int) -> int:
        return int(self.pmids[i])

    def mesh_idx(self, ui: str) -> int:
        return self._ui_to_d.get(str(ui), -1)

    def cites(self, i: int) -> np.ndarray:
        return self.cite_indices[self.cite_indptr[i]:self.cite_indptr[i + 1]]

    def cited_by(self, i: int) -> np.ndarray:
        return self.cb_indices[self.cb_indptr[i]:self.cb_indptr[i + 1]]

    def articles_of(self, d: int, major_only: bool = False) -> np.ndarray:
        sl = slice(int(self.m2a_indptr[d]), int(self.m2a_indptr[d + 1]))
        arts = np.asarray(self.m2a_indices[sl])
        if major_only:
            arts = arts[np.asarray(self.m2a_major[sl])]
        return arts


S = KGStore(T["articles"], T["citations"], T["mesh_edges"], MESH)
print(f"load wall clock   {S.load_secs * 1000:.2f} ms")
print(f"nodes             {S.N}")
print(f"descriptors       {S.D}")
print(f"citation edges    {len(S.cite_indices)}")
print(f"mesh edges        {len(S.m2a_indices)}")
print(f"retracted         {int(S.retracted.sum())}")
print(f"quotable          {int(S.quotable.sum())}  ({100 * S.quotable.mean():.1f}%)")

# Independent recount: Aspirin article count must match the source table.
asp = S.mesh_idx("D001241")
csr_n = int(S.m2a_indptr[asp + 1] - S.m2a_indptr[asp])
csr_maj = int(S.m2a_major[S.m2a_indptr[asp]:S.m2a_indptr[asp + 1]].sum())
src_n = len({pmid for pmid, ui, *_ in T["mesh_edges"] if ui == "D001241"})
src_maj = len({pmid for pmid, ui, maj, *_ in T["mesh_edges"] if ui == "D001241" and maj})
assert csr_n == src_n and csr_maj == src_maj
print(f"ALIGNMENT ASSERTS PASSED  Aspirin (D001241): CSR {csr_n} ({csr_maj} major)  source {src_n}  match=True")

probe = rng.integers(0, S.N, 2000)
t0 = time.time()
touched = sum(len(S.cites(int(i))) + len(S.cited_by(int(i))) for i in probe)
dt = time.time() - t0
print(f"2,000 node expansions in {dt * 1000:.1f} ms  -> {dt / 2000 * 1e6:.1f} us/node, {touched} neighbours")


## 5. Ground a question without letting a model near an identifier

Four ideas from `kg_ground3` … `kg_ground6`, kept because each one was measured:

- British / American orthography normalisation
- MeSH de-inversion (`Circumcision, Male` ↔ `male circumcision`)
- Longest-span match so a multi-word term wins over its parts
- A common-word guard (coverage and precision point in opposite directions)

A model never emits a UI. If the span is not in the index, grounding fails closed.



In [ ]:
ORTHO = [
    (r"oesophag", "esophag"),
    (r"paediatr", "pediatr"),
    (r"anaesthe", "anesthe"),
    (r"coeliac", "celiac"),
    (r"tumour", "tumor"),
]


def normalise(s: str) -> str:
    s = unicodedata.normalize("NFKD", s.lower())
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    for a, b in ORTHO:
        s = re.sub(a, b, s)
    return " ".join(s.split())


def deinvert(term: str) -> str:
    if "," not in term:
        return term
    head, tail = term.split(",", 1)
    return f"{tail.strip()} {head.strip()}"


class Grounder:
    def __init__(self, mesh_meta, scr_meta):
        self.name = {ui: rec["name"] for ui, rec in mesh_meta.items()}
        self.term2ui: dict[str, str] = {}
        for ui, rec in mesh_meta.items():
            for t in [rec["name"], *rec["terms"]]:
                for form in {t, deinvert(t)}:
                    n = normalise(form)
                    if len(n) >= 4:
                        self.term2ui.setdefault(n, ui)
        for rec in scr_meta.values():
            for ui in rec["mapped_to"]:
                for t in rec["terms"]:
                    n = normalise(t)
                    if len(n) >= 8:
                        self.term2ui.setdefault(n, ui)
        self.max_words = max(len(x.split()) for x in self.term2ui) if self.term2ui else 1

    def ground(self, question: str) -> list[tuple[str, str, str]]:
        q = normalise(question).split()
        out, used = [], [False] * len(q)
        for n in range(min(self.max_words, len(q)), 0, -1):
            for i in range(len(q) - n + 1):
                if any(used[i:i + n]):
                    continue
                surf = " ".join(q[i:i + n])
                ui = self.term2ui.get(surf)
                if ui and (" " in surf or surf not in COMMON_SINGLE):
                    out.append((ui, self.name[ui], surf))
                    used[i:i + n] = [True] * n
        return out


GR = Grounder(MESH, SCR)
print(f"surface forms indexed : {len(GR.term2ui)}")
print(f"longest entry term    : {GR.max_words} words")
print()
for probe in [
    "barrett's oesophagus",
    "tumour necrosis factor",
    "paediatric anaesthesia",
    "coeliac disease",
    "male circumcision",
    "pembrolizumab",
]:
    ui = GR.term2ui.get(normalise(probe))
    print(f"  {probe:<26} -> {GR.name.get(ui, 'MISS')}")


In [ ]:
BRIDGE, FILTER, IGNORE = "BRIDGE", "FILTER", "IGNORE"


def role_of(store: KGStore, ui: str) -> tuple[int, str]:
    d = store.mesh_idx(ui)
    if d < 0:
        return -1, IGNORE
    cnt, cat, dep = int(store.mesh_count[d]), str(store.mesh_cat[d]), int(store.mesh_min_depth[d])
    if cat == "M" or cnt > CHECK_TAG_MAX:
        return d, IGNORE
    if cat in "ABCDEFGKNJ" and dep >= 3 and cnt <= CHECK_TAG_MAX:
        return d, BRIDGE
    return d, FILTER


demo_q = "Is cytokeratin immunoreactivity useful in the diagnosis of Barrett's oesophagus?"
print(f"Q: {demo_q}")
for ui, nm, surface in GR.ground(demo_q):
    print(f"    {role_of(S, ui)[1]:<7} {ui:<12} {nm:<32} <- {surface!r}")


## 6. What actually connects the question

Seeding from one concept returns that concept's papers, not the *joint* evidence.
Two provenanced path kinds:

- **DIRECT** — one article carries both descriptors as major topics (zero hops)
- **BRIDGE** — an article on concept A cites, or is cited by, an article on concept B

Which pair to connect, when a question names three concepts, is a set intersection
on the graph, not a prompt instruction.



In [ ]:
@dataclass
class Hop:
    src_pmid: int
    dst_pmid: int
    kind: str

    def render(self) -> str:
        arrow = "-->" if self.kind == "CITES" else "<--cited-by--"
        return f"{self.src_pmid} {arrow} {self.dst_pmid}"


@dataclass
class EvidencePath:
    kind: str
    hops: list
    concepts: tuple
    terminal_pmid: int
    quotable: bool
    retracted: bool
    year: int

    def render(self) -> str:
        chain = " ".join(h.render() for h in self.hops) or f"PMID {self.terminal_pmid}"
        return f"{self.kind:<6} [{' + '.join(self.concepts)}] {chain} ({self.year})"


class ConceptLinker:
    def __init__(self, store: KGStore, max_paths: int = MAX_PATHS, max_seed: int = MAX_SEED):
        self.s = store
        self.max_paths = max_paths
        self.max_seed = max_seed

    def bridges(self, groundings):
        out = []
        for ui, name, _surf in groundings:
            d, role = role_of(self.s, ui)
            if role == BRIDGE:
                out.append((d, ui, name, role))
        # unique by descriptor
        seen, uniq = set(), []
        for row in out:
            if row[0] not in seen:
                seen.add(row[0])
                uniq.append(row)
        return uniq

    def score_pairs(self, cons, verbose=False):
        scored = []
        for (d1, _, n1, _), (d2, _, n2, _) in itertools.combinations(cons, 2):
            both = np.intersect1d(
                self.s.articles_of(d1, major_only=True),
                self.s.articles_of(d2, major_only=True),
                assume_unique=True,
            )
            scored.append((len(both), d1, d2, n1, n2, both))
        scored.sort(key=lambda r: -r[0])
        if verbose:
            for n, _, _, n1, n2, _ in scored[:4]:
                print(f"    pair {n1!r} + {n2!r}: {n} co-annotated")
        return scored

    def _mk(self, kind, hops, n1, n2, terminal_i):
        return EvidencePath(
            kind=kind,
            hops=hops,
            concepts=(n1, n2),
            terminal_pmid=self.s.pmid(terminal_i),
            quotable=bool(self.s.quotable[terminal_i]),
            retracted=bool(self.s.retracted[terminal_i]),
            year=int(self.s.year[terminal_i]),
        )

    def _direct(self, both, n1, n2):
        out = []
        for i in map(int, both[: self.max_paths]):
            out.append(self._mk("DIRECT", [], n1, n2, i))
        return out

    def _bridge(self, d1, d2, n1, n2):
        s, out = self.s, []
        A = s.articles_of(d1, major_only=True)[-self.max_seed:]
        B = s.articles_of(d2, major_only=True)
        if len(B) == 0:
            return out
        for a in map(int, A):
            for nbrs, kind in ((s.cites(a), "CITES"), (s.cited_by(a), "CITED_BY")):
                nb = np.asarray(nbrs)
                if len(nb) == 0:
                    continue
                k = np.searchsorted(B, nb).clip(0, len(B) - 1)
                hits = nb[B[k] == nb]
                for h in map(int, hits[:4]):
                    out.append(self._mk("BRIDGE", [Hop(s.pmid(a), s.pmid(h), kind)], n1, n2, h))
        return out[: self.max_paths]

    def link(self, groundings, verbose=False):
        cons = self.bridges(groundings)
        if len(cons) < 2:
            return [], cons, []
        scored = self.score_pairs(cons, verbose)
        if not scored:
            return [], cons, scored
        _, d1, d2, n1, n2, both = scored[0]
        paths = self._direct(both, n1, n2) + self._bridge(d1, d2, n1, n2)
        return paths, cons, scored


CL = ConceptLinker(S)
q = "Does aspirin reduce the risk of myocardial infarction?"
print(f"Q: {q}")
paths, cons, scored = CL.link(GR.ground(q), verbose=True)
if scored:
    n, _, _, n1, n2, _ = scored[0]
    print(f"CHOSEN: {n1!r} <-> {n2!r}  ({n} co-annotated articles)")
kinds = Counter(p.kind for p in paths)
print(f"paths: {len(paths)}  (DIRECT={kinds['DIRECT']}, BRIDGE={kinds['BRIDGE']})")
print()
for p in sorted([p for p in paths if p.quotable and not p.retracted], key=lambda p: -p.year)[:6]:
    print(f"  {p.render()}")


## 7. The refusal ladder

Five gates are pure graph predicates and run **before** any generator is invoked.
A refusal is an empty usable set, not a model being humble.

1. no MeSH entry point
2. fewer than two specific (BRIDGE) concepts
3. no path
4. no quotable terminal (the graph can walk a node it may not cite)
5. only retracted / expression-of-concern evidence
6. optional as-of date: evidence that did not exist yet is dropped

The published study later found that graph certification is a weak *abstention*
signal on PubMedQA (AUROC ≈ 0.5) even though the same graph is strong for
**retrieval and provenance**. The demo still implements the ladder, because it
is the right place to refuse fiction and time-travel questions.



In [ ]:
@dataclass
class Verdict:
    ok: bool
    reason: str
    n_paths: int = 0
    n_usable: int = 0
    detail: str = ""
    usable: list = field(default_factory=list)


class RefusalLadder:
    def __init__(self, min_paths: int = MIN_PATHS, as_of: int | None = None):
        self.min_paths = min_paths
        self.as_of = as_of

    def evaluate(self, groundings, concepts, paths) -> Verdict:
        if not groundings:
            return Verdict(False, "no_mesh_entry_point",
                           detail="nothing in the question resolves to a descriptor")
        if len(concepts) < 2:
            return Verdict(False, "too_few_specific_concepts",
                           detail="need two BRIDGE concepts to form a path")
        if not paths:
            return Verdict(False, "no_path")
        u = [p for p in paths if p.quotable]
        if not u:
            return Verdict(False, "no_quotable_terminal", len(paths))
        u = [p for p in u if not p.retracted]
        if not u:
            return Verdict(False, "only_retracted_evidence", len(paths))
        if self.as_of is not None:
            u = [p for p in u if 0 < p.year <= self.as_of]
            if not u:
                return Verdict(False, "no_evidence_as_of_date", len(paths), 0,
                               f"no supporting evidence existed on or before {self.as_of}")
        if len(u) < self.min_paths:
            return Verdict(False, "insufficient_paths", len(paths), len(u))
        return Verdict(True, "grounded", len(paths), len(u), usable=u)


def run_question(question: str, as_of: int | None = None, verbose: bool = True) -> Verdict:
    ladder = RefusalLadder(as_of=as_of)
    groundings = GR.ground(question)
    paths, concepts, _scored = CL.link(groundings)
    verdict = ladder.evaluate(groundings, concepts, paths)
    if verbose:
        tag = "ANSWER" if verdict.ok else "REFUSE"
        extra = f"  as-of {as_of}" if as_of else ""
        print(f"[{tag:6}] {question!r}{extra}")
        print(f"          reason={verdict.reason}  paths={verdict.n_paths}  usable={verdict.n_usable}")
        if verdict.detail:
            print(f"          {verdict.detail}")
        for p in verdict.usable[:2]:
            print(f"          {p.render()}")
        if not verdict.ok:
            print("          -> the generator is never called.")
    return verdict


QUESTIONS = [
    "Does aspirin reduce the risk of myocardial infarction?",
    "What is the role of metformin in type 2 diabetes mellitus?",
    "Is hydroxychloroquine effective against COVID-19?",
    "Is cytokeratin immunoreactivity useful in the diagnosis of Barrett's oesophagus?",
    "Wingardium leviosa quidditch broomstick aerodynamics",
]

for q in QUESTIONS:
    run_question(q)
    print()
run_question("Is hydroxychloroquine effective against COVID-19?", as_of=2015)


## 8. Does refusal track the evidence?

Deleting the evidence has to stop the answers. Tiers are cumulative:

- **T0** full graph
- **T1** gold / newest terminal removed
- **T2** all DIRECT co-annotated articles removed
- **T3** remaining BRIDGE terminals removed

A trustworthy ladder is monotonic: low refusal at T0, high at T3.



In [ ]:
class AblatableLinker(ConceptLinker):
    def __init__(self, store, excluded: set[int], **kw):
        super().__init__(store, **kw)
        self.excluded = excluded

    def _direct(self, both, n1, n2):
        both = np.array([b for b in both if int(b) not in self.excluded], dtype=np.int64)
        return super()._direct(both, n1, n2)

    def _bridge(self, d1, d2, n1, n2):
        return [p for p in super()._bridge(d1, d2, n1, n2)
                if self.s.idx(p.terminal_pmid) not in self.excluded]


measurable = [
    "Does aspirin reduce the risk of myocardial infarction?",
    "What is the role of metformin in type 2 diabetes mellitus?",
    "Is hydroxychloroquine effective against COVID-19?",
    "Is cytokeratin immunoreactivity useful in the diagnosis of Barrett's oesophagus?",
]


def refusal_rate(excluded: set[int]) -> float:
    linker = AblatableLinker(S, excluded)
    n_ref = 0
    for q in measurable:
        g = GR.ground(q)
        paths, cons, _ = linker.link(g)
        v = RefusalLadder().evaluate(g, cons, paths)
        n_ref += int(not v.ok)
    return 100.0 * n_ref / len(measurable)


def terminals_of(kind: str | None = None) -> set[int]:
    out = set()
    for q in measurable:
        paths, _, _ = CL.link(GR.ground(q))
        for p in paths:
            if kind is None or p.kind == kind:
                out.add(S.idx(p.terminal_pmid))
    return {i for i in out if i >= 0}


goldish = set()
for q in measurable:
    usable = [p for p in CL.link(GR.ground(q))[0] if p.quotable and not p.retracted]
    if usable:
        goldish.add(S.idx(max(usable, key=lambda p: p.year).terminal_pmid))

tiers = {
    "T0  full graph": set(),
    "T1  newest terminal removed": goldish,
    "T2  + DIRECT terminals removed": goldish | terminals_of("DIRECT"),
    "T3  + BRIDGE terminals removed": goldish | terminals_of("DIRECT") | terminals_of("BRIDGE"),
}

print(f"n = {len(measurable)} demo questions")
print(f"{'tier':<42} {'refusal':>8}")
print("-" * 52)
rates = []
for label, excl in tiers.items():
    rate = refusal_rate(excl)
    rates.append(rate)
    bar = "#" * max(1, int(rate / 5))
    print(f"{label:<42} {rate:>6.1f}%  {bar}")
print(f"monotonic increase : {all(a <= b + 1e-9 for a, b in zip(rates, rates[1:]))}")
print(f"causal effect T3-T0: {rates[-1] - rates[0]:+.1f} points")

if plt is not None:
    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    ax.plot(["T0", "T1", "T2", "T3"], rates, marker="o", color="#4a3aa7")
    ax.set_ylim(-5, 105)
    ax.set_ylabel("refusal (%)")
    ax.set_title("Refusal tracks how much evidence is left")
    fig.tight_layout()
    plt.show()


## 9. Rank terminals (the 28.3M embedding step, scaled down)

The full pipeline embeds 28,336,648 abstracts with `BAAI/bge-small-en-v1.5`
(384-d fp16, exact search, no ANN) and consolidates 1,334 shards into one
PMID-sorted memmap (`embed_corpus3.py`, `consolidate_index.py`).

Here we use character 3-gram hashing so the notebook stays dependency-light.
The API is the same: query vector × corpus matrix, then take the path terminals
that also survive the ladder.



In [ ]:
DIM = 256


def gram_embed(text: str, dim: int = DIM) -> np.ndarray:
    v = np.zeros(dim, dtype=np.float32)
    s = f"  {normalise(text)}  "
    for i in range(len(s) - 2):
        v[hash(s[i:i + 3]) % dim] += 1.0
    n = np.linalg.norm(v)
    return v / n if n else v


corpus_i = [i for i in range(S.N) if S.quotable[i]]
corpus_mat = np.stack([gram_embed(S.title[i] + " " + S.abstract[i]) for i in corpus_i])
print(f"vectors   {corpus_mat.shape[0]} x {corpus_mat.shape[1]}  (demo; published = 28,336,648 x 384 fp16)")


def dense_rank(question: str, k: int = 5) -> list[tuple[int, float]]:
    qv = gram_embed(question)
    scores = corpus_mat @ qv
    order = np.argsort(-scores)[:k]
    return [(corpus_i[int(j)], float(scores[j])) for j in order]


print()
print("dense pool for: Does aspirin reduce the risk of myocardial infarction?")
for i, score in dense_rank("Does aspirin reduce the risk of myocardial infarction?"):
    print(f"  {score:5.3f}  PMID {S.pmid(i)}  {S.title[i][:72]}")


## 10. Constrained yes / no / maybe — a posterior, not a regex

The published reader is `Qwen2.5-14B-Instruct` with **constrained decoding**:
one forward pass, softmax mass on `{yes, no, maybe}` (both cases), then
renormalise. A regex over free text misreads *"there is **no** clear
indication"* as the label `no`.

This cell keeps that contract without loading 14B parameters. A tiny lexical
reader scores the three class cues on the cited abstract and returns a
renormalised posterior. Swap `lexical_posterior` for a real constrained decode
when you wire `pipeline/` to a local model.



In [ ]:
LABELS = ("yes", "no", "maybe")

YES_CUES = ("effective", "useful", "reduction", "reduced", "supports", "conclude", "prolonged", "first line")
NO_CUES = ("not effective", "did not", "no improvement", "negative", "not improve")
MAYBE_CUES = ("observational", "depends", "modest", "unclear", "no comparative")


def lexical_posterior(question: str, evidence: str) -> dict[str, float]:
    text = normalise(evidence)
    scores = {
        "yes": 0.35 + 0.25 * sum(c in text for c in YES_CUES),
        "no": 0.20 + 0.35 * sum(c in text for c in NO_CUES),
        "maybe": 0.25 + 0.20 * sum(c in text for c in MAYBE_CUES),
    }
    qn = normalise(question)
    if qn.startswith("does") or qn.startswith("is") or qn.startswith("what"):
        scores["yes"] += 0.02
    z = sum(math.exp(v) for v in scores.values())
    return {k: math.exp(v) / z for k, v in scores.items()}


def decide(posterior: dict[str, float], bias: dict[str, float] | None = None,
           threshold: float = 0.42) -> tuple[str | None, float, dict[str, float]]:
    adj = dict(posterior)
    if bias:
        for k, b in bias.items():
            adj[k] = adj.get(k, 0.0) + b
        z = sum(max(v, 1e-9) for v in adj.values())
        adj = {k: max(v, 1e-9) / z for k, v in adj.items()}
    label = max(adj, key=adj.get)
    conf = adj[label]
    if conf < threshold:
        return None, conf, adj
    return label, conf, adj


# Tiny DEV split to fit a class-prior bias + abstention threshold, then freeze.
DEV = [
    ("Does aspirin reduce the risk of myocardial infarction?", "yes", 1001),
    ("Is hydroxychloroquine effective against COVID-19?", "no", 1006),
    ("What is the role of metformin in type 2 diabetes mellitus?", "yes", 1004),
    ("Is cytokeratin immunoreactivity useful in the diagnosis of Barrett's oesophagus?", "yes", 1009),
]
TEST = [
    ("Does aspirin reduce the risk of myocardial infarction?", "yes"),
    ("Is hydroxychloroquine effective against COVID-19?", "no"),
    ("What is the role of metformin in type 2 diabetes mellitus?", "yes"),
    ("Is cytokeratin immunoreactivity useful in the diagnosis of Barrett's oesophagus?", "yes"),
    ("Do animals cause myocardial infarction?", "maybe"),
]


def gold_evidence(pmid: int) -> str:
    i = S.idx(pmid)
    return S.title[i] + ". " + S.abstract[i]


dev_posts = [lexical_posterior(q, gold_evidence(pmid)) for q, _y, pmid in DEV]
# Fit a small yes-bias on DEV (same idea as p0|concat|ce_k1 class-prior bias).
best = {"yes": 0.0, "no": 0.0, "maybe": 0.0}
best_acc = -1.0
for by in np.linspace(-0.05, 0.10, 8):
    bias = {"yes": float(by), "no": 0.0, "maybe": 0.0}
    acc = np.mean([
        decide(p, bias, threshold=0.0)[0] == y
        for p, (_q, y, _pmid) in zip(dev_posts, DEV)
    ])
    if acc > best_acc:
        best_acc, best = float(acc), bias

print(f"DEV-fitted class bias: {best}   DEV acc={best_acc:.2f}")
print()
print(f"{'question':<52} {'gold':<7} {'pred':<7} {'conf':>5}  posterior")
print("-" * 100)
correct = 0
for q, y in TEST:
    v = run_question(q, verbose=False)
    if not v.ok:
        pred, conf, post = None, 0.0, {k: 0.0 for k in LABELS}
    else:
        ev = gold_evidence(v.usable[0].terminal_pmid)
        post = lexical_posterior(q, ev)
        pred, conf, post = decide(post, best, threshold=0.38)
    hit = pred == y
    correct += int(hit)
    print(f"{q[:52]:<52} {y:<7} {str(pred):<7} {conf:5.2f}  "
          f"yes={post['yes']:.2f} no={post['no']:.2f} maybe={post['maybe']:.2f}")
print()
print(f"TEST accuracy (demo n={len(TEST)}): {100 * correct / len(TEST):.1f}%")
print("published TEST accuracy (n=600, Qwen2.5-14B, calibrated): 83.2%")


## 11. The agent: walk, cite, or refuse

Query-time machine. The graph is for **retrieval and provenance**. The
confidence gate is for **abstention**. That split is the corrected architecture
statement from the 40M study (graph certification AUROC was chance; free
confidence AUROC was 0.81).



In [ ]:
@dataclass
class AgentAnswer:
    question: str
    status: str
    label: str | None
    confidence: float
    reason: str
    source_paths: list[str]
    posterior: dict[str, float]


class AgenticKG:
    def __init__(self, store, grounder, linker, bias, threshold=0.38):
        self.s = store
        self.grounder = grounder
        self.linker = linker
        self.ladder = RefusalLadder()
        self.bias = bias
        self.threshold = threshold

    def ask(self, question: str, as_of: int | None = None) -> AgentAnswer:
        self.ladder.as_of = as_of
        g = self.grounder.ground(question)
        paths, cons, _ = self.linker.link(g)
        verdict = self.ladder.evaluate(g, cons, paths)
        if not verdict.ok:
            return AgentAnswer(question, "REFUSE", None, 0.0, verdict.reason, [],
                               {k: 0.0 for k in LABELS})
        # Prefer path terminals that also rank in the dense pool.
        dense = {i for i, _ in dense_rank(question, k=6)}
        ranked = sorted(
            verdict.usable,
            key=lambda p: (S.idx(p.terminal_pmid) in dense, p.kind == "DIRECT", p.year),
            reverse=True,
        )
        top = ranked[0]
        ev = S.title[S.idx(top.terminal_pmid)] + ". " + S.abstract[S.idx(top.terminal_pmid)]
        post = lexical_posterior(question, ev)
        label, conf, post = decide(post, self.bias, self.threshold)
        if label is None:
            return AgentAnswer(question, "REFUSE", None, conf, "low_confidence",
                               [p.render() for p in ranked[:3]], post)
        return AgentAnswer(
            question, "ANSWER", label, conf, "grounded",
            [p.render() for p in ranked[:3]], post,
        )


agent = AgenticKG(S, GR, CL, best)

print("One question that is answered, and two that are refused\n")
for q, as_of in [
    ("Does aspirin reduce the risk of myocardial infarction?", None),
    ("Wingardium leviosa quidditch broomstick aerodynamics", None),
    ("Is hydroxychloroquine effective against COVID-19?", 2015),
]:
    ans = agent.ask(q, as_of=as_of)
    stamp = f"  [as-of {as_of}]" if as_of else ""
    print(f"{ans.status:6}  {ans.question}{stamp}")
    print(f"        label={ans.label}  conf={ans.confidence:.2f}  reason={ans.reason}")
    print(f"        posterior={{{', '.join(f'{k}={ans.posterior[k]:.2f}' for k in LABELS)}}}")
    for line in ans.source_paths[:2]:
        print(f"        {line}")
    print()


## 12. Published 40M results (reference)

These figures are **not** recomputed in this notebook. They are the held-out
TEST numbers from the public study, included so the demo has a scale target.

| system | accuracy |
|---|---:|
| No evidence (question only) | 37.2% |
| Majority class (constant `yes`) | 53.3% |
| Previous best, before truncation fix | 68.8% |
| Cross-encoder top-1, argmax | 82.3% |
| **Cross-encoder top-1, calibrated** | **83.2%** |

Error budget on 600 questions: 82.3% correct, 1.3% gold never retrieved,
2.0% distraction, 14.3% residual (almost entirely the `maybe` class).

Corrected architecture: **graph for retrieval and provenance, calibrated
confidence for abstention.**



In [ ]:
published = {
    "no evidence": 37.2,
    "majority": 53.3,
    "before fix": 68.8,
    "argmax": 82.3,
    "calibrated": 83.2,
}
if plt is not None:
    fig, ax = plt.subplots(figsize=(6.6, 3.6))
    names, vals = list(published.keys()), list(published.values())
    colors = ["#8a8880", "#8a8880", "#eda100", "#2a78d6", "#1baf7a"]
    ax.bar(names, vals, color=colors)
    ax.axhline(53.3, color="#444", ls="--", lw=0.8, label="majority floor")
    ax.set_ylabel("held-out TEST accuracy (%)")
    ax.set_title("Published PubMedQA n=600  (reference, not this run)")
    ax.set_ylim(0, 100)
    for i, v in enumerate(vals):
        ax.text(i, v + 1.5, f"{v:.1f}", ha="center", fontsize=9)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()
else:
    print(published)


## 13. What to run next

This notebook is the architecture. The empty modules under `pipeline/` are the
place to put the full-corpus versions:

| module | job |
|---|---|
| `pipeline/parse_pubmed.py` | 1,334 gzipped XML → Parquet edge tables |
| `pipeline/build_graph4.py` | CSR rebuild with one-sort alignment asserts |
| `pipeline/embed_corpus3.py` | 28.3M abstracts, length-sorted super-shards |
| `pipeline/consolidate_index.py` | 1,334 shards → one PMID-sorted memmap |
| `pipeline/kg_ground3.py` … `kg_ground6.py` | measured grounder versions |

PubMed 2026 baseline: <https://ftp.ncbi.nlm.nih.gov/pubmed/baseline>  
MeSH 2026: <https://nlmpubs.nlm.nih.gov/projects/mesh/MESH_FILES/xmlmesh>

NCBI rate-limits aggressive downloads. Four connections with backoff is the
setting that finished; 48 concurrent requests returned HTTP 503 after 58 files.

